In [4]:
from datasets import load_dataset

def preview_remote_samples(num_samples=5):
    # Added 'synthetic_sft' as the config name
    dataset = load_dataset("microsoft/rStar-Coder", "synthetic_sft", split="train", streaming=True)

    count = 0
    print(f"Fetching {num_samples} Python samples from 'synthetic_sft'...\n")

    for entry in dataset:
        # Check available keys. rStar-Coder synthetic_sft usually uses 'question' and 'response'
        question = entry.get("question", "")
        # In rStar, the reasoning and code are often inside 'response'
        content = entry.get("response", entry.get("solution", ""))

        # Simple Python check
        if "def " in content or "import " in content:
            count += 1
            print(f"--- SAMPLE {count} ---")
            print(f"QUESTION:\n{question[:200]}...")
            print(f"\nCONTENT (Reasoning + Code):\n{content[:500]}...")
            print("-" * 50 + "\n")

        if count >= num_samples:
            break

preview_remote_samples(5)

Fetching 5 Python samples from 'synthetic_sft'...

--- SAMPLE 1 ---
QUESTION:
**Problem Description**:

A matrix of size `m x m` is called symmetrical if it remains the same when the rows are reversed. You are given `m^2` integers. Your task is to arrange these `m^2` integers i...

CONTENT (Reasoning + Code):
Okay, I need to solve this problem where I have to determine if a given set of m^2 integers can be arranged into an m x m symmetrical matrix. If possible, I have to output such a matrix, otherwise print "NO". 

First, let me understand the problem statement clearly. A symmetrical matrix here is one that remains the same when the rows are reversed. Wait, no—wait. Oh right, no. Wait, the definition says that when the rows are reversed, the matrix remains the same. So, for example, if the first row...
--------------------------------------------------

--- SAMPLE 2 ---
QUESTION:
**Problem Description**:
A baker has several trays of cookies, each with a certain number of cookies. Ever

In [5]:
from datasets import load_dataset

# Load the streaming dataset
ds = load_dataset("microsoft/rStar-Coder", "synthetic_sft", split="train", streaming=True)

# Get the first item
first_row = next(iter(ds))

print("Available Fields:")
for key in first_row.keys():
    print(f"- {key}")

print("\n--- Value Preview ---")
print(f"Sample from 'question': {str(first_row.get('question'))[:100]}...")

Available Fields:
- question
- seed_question
- seed_source
- response
- code

--- Value Preview ---
Sample from 'question': **Problem Description**:
A library is preparing reading materials for *n* book clubs. Each book club...


In [6]:
from datasets import load_dataset

def inspect_one_sample():
    # Loading synthetic_sft in streaming mode
    dataset = load_dataset("microsoft/rStar-Coder", "synthetic_sft", split="train", streaming=True)

    for entry in dataset:
        q = entry.get("question", "")
        r = entry.get("response", "")
        c = entry.get("code", "")

        # Ensure we get a Python sample with actual reasoning
        if "def " in c and len(r) > 50:
            print("=== RAW SAMPLE PREVIEW ===")
            print(f"SOURCE: {entry.get('seed_source', 'Unknown')}")
            print("-" * 30)
            print(f"QUESTION (First 200 chars):\n{q[:200]}...")
            print("-" * 30)
            print(f"REASONING (The 'Response' field):\n{r}")
            print("-" * 30)
            print(f"CODE (The 'Code' field):\n{c}")
            print("-" * 30)

            # Metadata for our Curriculum/Conciseness logic
            print("=== STATS FOR FILTERING ===")
            print(f"Reasoning Word Count: {len(r.split())}")
            print(f"Code Word Count: {len(c.split())}")
            break

inspect_one_sample()


=== RAW SAMPLE PREVIEW ===
SOURCE: codeforces
------------------------------
QUESTION (First 200 chars):
**Problem Description**:

A matrix of size `m x m` is called symmetrical if it remains the same when the rows are reversed. You are given `m^2` integers. Your task is to arrange these `m^2` integers i...
------------------------------
REASONING (The 'Response' field):
Okay, I need to solve this problem where I have to determine if a given set of m^2 integers can be arranged into an m x m symmetrical matrix. If possible, I have to output such a matrix, otherwise print "NO". 

First, let me understand the problem statement clearly. A symmetrical matrix here is one that remains the same when the rows are reversed. Wait, no—wait. Oh right, no. Wait, the definition says that when the rows are reversed, the matrix remains the same. So, for example, if the first row is [a, b, c], the last row should be [a, b, c] as well when reversed. Wait, no, wait. Let me think: for a matrix of size m x 

In [7]:
import json
from datasets import load_dataset

def extract_stratified_rstar(total_goal=16000):
    # Limits based on your 30-40-30 request
    limits = {
        "easy": int(total_goal * 0.30),   # 4800 samples < 3000 tokens
        "medium": int(total_goal * 0.40), # 6400 samples < 5000 tokens
        "hard": int(total_goal * 0.30)    # 4800 samples < 7000 tokens
    }

    buckets = {"easy": [], "medium": [], "hard": []}

    # Stream the SFT portion of rStar
    print("Streaming dataset... this may take a moment to initialize.")
    ds = load_dataset("microsoft/rStar-Coder", "synthetic_sft", split="train", streaming=True)

    count = 0
    for entry in ds:
        q = entry.get("question", "")
        r = entry.get("response", "")
        c = entry.get("code", "")

        # 1. Python Filter
        if "def " not in c and "import " not in c:
            continue

        # 2. Token Approximation (Words to Tokens)
        # Using a conservative 1 word = 1.3 tokens for code/reasoning
        approx_tokens = int(len((r + c).split()) * 1.3)

        # 3. Stratification Logic
        if approx_tokens < 3000:
            if len(buckets["easy"]) < limits["easy"]:
                buckets["easy"].append({"question": q, "response": r, "code": c, "tokens": approx_tokens})
        elif approx_tokens < 5000:
            if len(buckets["medium"]) < limits["medium"]:
                buckets["medium"].append({"question": q, "response": r, "code": c, "tokens": approx_tokens})
        elif approx_tokens < 7000:
            if len(buckets["hard"]) < limits["hard"]:
                buckets["hard"].append({"question": q, "response": r, "code": c, "tokens": approx_tokens})

        # Progress update every 1000 items
        current_total = sum(len(v) for v in buckets.values())
        if current_total % 1000 == 0 and current_total > count:
            print(f"Progress: {current_total}/{total_goal} collected...")
            count = current_total

        if all(len(buckets[k]) >= limits[k] for k in buckets):
            break

    # 4. Curriculum Sorting
    # We combine them and sort by token count to ensure the model learns
    # simple logic before complex/long logic.
    final_data = buckets["easy"] + buckets["medium"] + buckets["hard"]
    final_data.sort(key=lambda x: x['tokens'])

    # 5. Save to JSONL
    with open('rstar_stratified_16k.jsonl', 'w') as f:
        for item in final_data:
            f.write(json.dumps(item) + '\n')

    print(f"\nSuccess! Total collected: {len(final_data)}")
    print(f"Easy (<3k): {len(buckets['easy'])} | Medium (<5k): {len(buckets['medium'])} | Hard (<7k): {len(buckets['hard'])}")

# Run the extractor
extract_stratified_rstar()

Streaming dataset... this may take a moment to initialize.
Progress: 1000/16000 collected...
Progress: 2000/16000 collected...
Progress: 3000/16000 collected...
Progress: 4000/16000 collected...
Progress: 5000/16000 collected...
Progress: 6000/16000 collected...
Progress: 7000/16000 collected...
Progress: 8000/16000 collected...
Progress: 9000/16000 collected...
Progress: 10000/16000 collected...
Progress: 11000/16000 collected...
Progress: 12000/16000 collected...
Progress: 13000/16000 collected...
Progress: 14000/16000 collected...
Progress: 15000/16000 collected...
Progress: 16000/16000 collected...

Success! Total collected: 16000
Easy (<3k): 4800 | Medium (<5k): 6400 | Hard (<7k): 4800


In [8]:
import json

def perform_sanity_check(file_path):
    issues = []
    stats = {"easy": 0, "medium": 0, "hard": 0, "total": 0}

    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            try:
                data = json.loads(line)
                stats["total"] += 1

                # 1. Missing Fields Check
                if not all(key in data for key in ['question', 'response', 'code']):
                    issues.append(f"Row {i}: Missing essential fields.")

                # 2. Python Check
                code = data.get('code', '')
                if "def " not in code and "import " not in code and "print(" not in code:
                    # Some very simple competitive programming logic might lack these,
                    # but usually, it's a sign of non-python or empty code.
                    if len(code.strip()) < 5:
                        issues.append(f"Row {i}: Code field looks empty or non-Python.")

                # 3. Token/Length Range Check
                tokens = data.get('tokens', 0)
                if tokens < 3000: stats["easy"] += 1
                elif tokens < 5000: stats["medium"] += 1
                elif tokens < 7000: stats["hard"] += 1

                # 4. Spot Check a random middle sample
                if i == 8000:
                    print("--- SPOT CHECK (Row 8000) ---")
                    print(f"Tokens: {tokens}")
                    print(f"Reasoning Preview: {data['response'][:150]}...")
                    print(f"Code Preview: {data['code'][:100]}...\n")

            except Exception as e:
                issues.append(f"Row {i}: JSON Error - {str(e)}")

    print("=== SANITY CHECK REPORT ===")
    print(f"Total Rows Scanned: {stats['total']}")
    print(f"Easy Samples Found: {stats['easy']}")
    print(f"Medium Samples Found: {stats['medium']}")
    print(f"Hard Samples Found: {stats['hard']}")

    if not issues:
        print("\n✅ PASSED: Data structure and stratification are correct.")
    else:
        print(f"\n⚠️ WARNING: Found {len(issues)} potential issues.")
        for issue in issues[:5]: # Show first 5 issues
            print(issue)

# Run it
perform_sanity_check('rstar_stratified_16k.jsonl')

--- SPOT CHECK (Row 8000) ---
Tokens: 4009
Reasoning Preview: <think>
Okay, I need to solve this programming problem. Let's read the problem statement carefully.

The task is: Given an array of N integers and an ...
Code Preview: from collections import Counter

n, d = map(int, input().split())
arr = list(map(int, input().split(...

=== SANITY CHECK REPORT ===
Total Rows Scanned: 16000
Easy Samples Found: 4800
Medium Samples Found: 6400
Hard Samples Found: 4800

✅ PASSED: Data structure and stratification are correct.


In [9]:
import json

def finalize_dataset_logic(input_file='rstar_stratified_16k.jsonl'):
    # Global Buckets
    buckets = {"EASY": [], "MEDIUM": [], "HARD": []}

    # Expanded Keyword Dictionary
    keywords = {
        "DP": ["dynamic programming", "knapsack", "memoization", "recursion", "lcs", "lis", "bitmask", "subsequence", "transition"],
        "GRAPH": ["dfs", "bfs", "dijkstra", "tree", "graph", "shortest path", "mst", "kruskal", "topological", "flow", "bellman-ford", "prim", "adjacency", "cycle-finding"],
        "MATH": ["prime", "modulo", "gcd", "combinatorics", "math", "number theory", "probability", "geometry", "sieve", "prefix sum"],
        "DATA_STRUCTURES": ["segment tree", "fenwick", "dsu", "union find", "priority queue", "heap", "trie", "stack", "queue", "deque", "linked list", "monotonic", "bst", "avl"],
        "SEARCH_SORT": ["binary search", "greedy", "sorting", "two pointers", "sliding window", "ternary search", "quicksort", "mergesort", "huffman"],
        "RECURSION_BACKTRACKING": ["backtracking", "n-queens", "sudoku", "permutations", "combinations", "subset sum"]
    }

    print("Reading and Tagging Data...")
    with open(input_file, 'r') as f:
        for line in f:
            item = json.loads(line)
            q, r, c, tokens = item['question'], item['response'], item['code'], item['tokens']

            # 1. Determine Difficulty
            if tokens < 3000: diff = "EASY"
            elif tokens < 5000: diff = "MEDIUM"
            else: diff = "HARD"

            # 2. Determine Categories
            content = (q + " " + r).lower()
            found = [tag for tag, keys in keywords.items() if any(k in content for k in keys)]
            if not found: found = ["IMPLEMENTATION"]
            found.append(diff)

            # 3. Format Entry
            tag_str = f"<tags> {', '.join(found)} </tags>"
            formatted_entry = {
                "instruction": q,
                "output": f"{tag_str}\n<think>\n{r.strip()}\n</think>\n<code>\n{c.strip()}\n</code>",
                "tokens": tokens,
                "difficulty": diff
            }
            buckets[diff].append(formatted_entry)

    # 4. Partitioning Logic (30-40-30 distribution for each set)
    # Stage 1: 12,000 (3.6k E, 4.8k M, 3.6k H)
    # Stage 2: 2,000 (600 E, 800 M, 600 H)
    # Stage 3: 2,000 (600 E, 800 M, 600 H)

    sets = {"stage1": [], "stage2": [], "stage3": []}

    for diff, size_1, size_2, size_3 in [("EASY", 3600, 600, 600),
                                         ("MEDIUM", 4800, 800, 800),
                                         ("HARD", 3600, 600, 600)]:
        data = buckets[diff]
        sets["stage1"].extend(data[:size_1])
        sets["stage2"].extend(data[size_1 : size_1 + size_2])
        sets["stage3"].extend(data[size_1 + size_2 : size_1 + size_2 + size_3])

    # 5. Curriculum Sort only Stage 1
    sets["stage1"].sort(key=lambda x: x['tokens'])

    # 6. Save Files
    for name, data in sets.items():
        filename = f"{name}_train.jsonl"
        with open(filename, 'w') as f:
            for item in data:
                # Remove internal 'tokens' and 'difficulty' keys before saving
                clean_item = {"instruction": item["instruction"], "output": item["output"]}
                f.write(json.dumps(clean_item) + '\n')
        print(f"Saved {filename} with {len(data)} samples.")

finalize_dataset_logic()

Reading and Tagging Data...
Saved stage1_train.jsonl with 12000 samples.
Saved stage2_train.jsonl with 2000 samples.
Saved stage3_train.jsonl with 2000 samples.


In [10]:
import json

def check_random_sample(filename='stage1_train.jsonl'):
    print(f"--- Checking Sample from {filename} ---\n")
    with open(filename, 'r') as f:
        # We skip to the end of the file to see a 'Hard' sample
        # because the file is curriculum-sorted!
        lines = f.readlines()
        sample = json.loads(lines[-1]) # Get the very last (hardest) sample

        print("PROMPT (Instruction):")
        print(f"{sample['instruction'][:300]}...\n")

        print("FORMATTED OUTPUT:")
        # We'll print the first 1000 characters of the output to see tags and start of thinking
        print(sample['output'][:1000])
        print("\n... [rest of reasoning and code] ...")

check_random_sample()

--- Checking Sample from stage1_train.jsonl ---

PROMPT (Instruction):
Problem Description:
Alice has two integers X and Y. In one operation, she can choose any integer d and perform one of the following two moves:
- Multiply X by d and divide Y by d.
- Multiply Y by d and divide X by d.

Alice can perform as many operations as she wants. Can she make X and Y equal?

I...

FORMATTED OUTPUT:
<tags> GRAPH, MATH, HARD </tags>
<think>
Okay, let's see. So the problem is Alice has two integers X and Y, and she can perform operations where she picks a number d and either multiply X by d and divide Y by d, or vice versa. The question is whether she can make X and Y equal after any number of these operations. Hmm, right.

Hmm. So the key is figuring out under what conditions this is possible. Let's think. Each operation, when you choose d, you're essentially moving factors between X and Y. Because multiplying X by d and dividing Y by d would transfer the factors of d from Y to X. Or the other w